# GROUP 2 — TELECOM CUSTOMER CHURN
## Data Preparation Notebook

**Prepared by:** Simon Nnaji  
**Role:** Data Preparation Lead  
**Project:** Group 2 — Telecom Customer Churn  
**Dataset:** `Customer_churn.csv`  
**Stage:** Data Preparation

### Purpose

This notebook documents the complete Data Preparation stage from raw customer data to a reproducible machine-learning-ready feature pipeline.

The preparation stage is responsible for:

1. validating the upstream raw-data baseline;
2. handling missing, blank and malformed values;
3. correcting datatypes;
4. handling identifiers;
5. defining feature groups;
6. encoding categorical variables;
7. scaling numerical variables where appropriate;
8. preventing data leakage;
9. splitting the dataset correctly;
10. building a reproducible preprocessing pipeline;
11. validating the transformed output;
12. handing a controlled feature matrix and preprocessing pipeline to Model Development.

**Core engineering rule:** the raw dataset is never overwritten, and transformations used for modelling must be reproducible.

## 1. Upstream Data Understanding Baseline

The supplied raw dataset contains **7,043 rows and 21 columns**.

The upstream assessment established:

- Exact duplicate rows: **0**
- Duplicate `customerID` values: **0**
- Pandas-recognised null values: **0**
- Whitespace-only `TotalCharges` values: **11**
- `TotalCharges` is stored as text.
- The 11 whitespace-only `TotalCharges` records occur at `tenure = 0` and `Churn = No`.

The preparation stage now converts those findings into explicit, reproducible technical decisions.

## 2. Import Libraries

These libraries support data manipulation, preprocessing, train/test splitting and validation.

In [1]:
import pandas as pd  # Import pandas for DataFrame operations and data validation.
import numpy as np  # Import NumPy for numerical operations.
from sklearn.compose import ColumnTransformer  # Apply different preprocessing rules to different feature groups.
from sklearn.impute import SimpleImputer  # Provide reproducible missing-value handling.
from sklearn.pipeline import Pipeline  # Chain preprocessing steps into one reproducible object.
from sklearn.preprocessing import OneHotEncoder  # Encode categorical variables into machine-readable columns.
from sklearn.preprocessing import StandardScaler  # Standardise numerical features where required.
from sklearn.model_selection import train_test_split  # Split features and target into training and test sets.

### What was done and why

The imports are intentionally focused on preparation.

`ColumnTransformer` and `Pipeline` are particularly important because they prevent the team from manually repeating transformations in different places. The same fitted preprocessing object can later be used on validation/test data and on new customer inputs in the prototype.

No model is trained in this notebook.

## 3. Load the Raw Dataset

The source file is loaded and immediately copied so that the raw source remains preserved.

In [2]:
file_path = "/content/Customer_churn.csv"  # Define the raw CSV path used in Google Colab.
raw_df = pd.read_csv(file_path)  # Load the raw CSV exactly as supplied.
df = raw_df.copy()  # Create a working copy so raw_df remains unchanged.

### What was done and why

The raw dataset is preserved in `raw_df`.

All subsequent preparation work is performed on `df` or through preprocessing pipelines. This creates a clear distinction between:

- **raw source data** — what was received;
- **prepared data** — what was derived;
- **preprocessing pipeline** — the reproducible rules connecting the two.

This separation is essential for auditability and debugging.

## 4. Validate Raw Shape and Schema

Before changing the data, record the initial structure.

In [3]:
print("Rows:", df.shape[0])  # Report the number of customer records.
print("Columns:", df.shape[1])  # Report the number of fields.
print("Column names:")  # Add a readable label before the schema.
print(df.columns.tolist())  # Display the complete original column order.

Rows: 7043
Columns: 21
Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


### What was done and why

The before-state is recorded so that the team can later verify that preparation did not accidentally remove or create records/fields.

Any intentional structural change must have a documented reason.

## 5. Validate Exact Duplicates

Duplicates are checked before feature construction.

In [4]:
duplicate_rows = int(df.duplicated().sum())  # Count exact duplicate records.
print("Exact duplicate rows:", duplicate_rows)  # Report the result.
assert duplicate_rows == 0, "Duplicate rows detected. Review before continuing."  # Stop if the upstream dataset differs unexpectedly.

Exact duplicate rows: 0


### What was done and why

The upstream data-understanding assessment found no exact duplicate rows. The assertion turns that observation into a safeguard.

If the wrong dataset version is loaded later, the notebook should fail visibly rather than silently producing a different model-training dataset.

## 6. Validate Customer ID Uniqueness

`customerID` is an identifier, so repeated IDs are checked separately.

In [5]:
duplicate_ids = int(df["customerID"].duplicated().sum())  # Count repeated customer IDs.
unique_ids = int(df["customerID"].nunique())  # Count distinct customer IDs.
print("Unique customer IDs:", unique_ids)  # Report distinct identifiers.
print("Duplicate customer IDs:", duplicate_ids)  # Report repeated identifiers.

Unique customer IDs: 7043
Duplicate customer IDs: 0


### What was done and why

The identifier is checked because two rows sharing one customer ID could represent repeated observations, data duplication or a different data-grain problem.

For this supplied extract, customer IDs are unique.

`customerID` will be retained temporarily for traceability but excluded from the model features.

## 7. Inspect Missing Values Before Treatment

Formal null detection is repeated at the preparation stage.

In [6]:
missing_by_column = df.isna().sum()  # Count pandas-recognised missing values by field.
display(missing_by_column.sort_values(ascending=False).to_frame("Missing values"))  # Display missing counts from highest to lowest.
print("Total missing values:", int(missing_by_column.sum()))  # Report the dataset-wide total.

,Missing values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Total missing values: 0


### What was done and why

There are **0 pandas-recognised null values** in the raw file.

However, the upstream assessment found **11 whitespace-only `TotalCharges` values**. Therefore formal null detection alone is insufficient.

The next step normalises whitespace so that these hidden missing values can be handled explicitly.

## 8. Normalise Whitespace in `TotalCharges`

Whitespace is converted to an explicit missing representation in the working copy.

In [7]:
df["TotalCharges"] = df["TotalCharges"].astype(str).str.strip()  # Convert the field to strings and remove leading/trailing whitespace.
df["TotalCharges"] = df["TotalCharges"].replace("", np.nan)  # Convert empty strings created by stripping into explicit missing values.
print("TotalCharges missing after whitespace normalisation:", int(df["TotalCharges"].isna().sum()))  # Verify the hidden blanks are now visible as missing.

TotalCharges missing after whitespace normalisation: 11


### What was done and why

The raw `TotalCharges` problem is not fixed by simply calling `isnull()` because the affected cells contain whitespace.

This transformation makes the missing state explicit.

The original raw file remains untouched; only the working copy is changed.

The expected result is **11 missing `TotalCharges` values**.

## 9. Convert `TotalCharges` to Numeric

The cleaned text field is converted to its intended numerical representation.

In [8]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # Convert valid charge strings to numbers and keep invalid values as missing.
print("TotalCharges dtype:", df["TotalCharges"].dtype)  # Confirm the new datatype.
print("TotalCharges missing after conversion:", int(df["TotalCharges"].isna().sum()))  # Confirm no unexpected conversion failures occurred.

TotalCharges dtype: float64
TotalCharges missing after conversion: 11


### What was done and why

`TotalCharges` represents cumulative financial expenditure, so it should be numerical for downstream modelling.

The conversion is deliberately performed **after** whitespace normalisation.

At this point the 11 problematic records remain explicitly missing. We have not yet decided whether they should become zero or receive another treatment. That decision needs business/technical justification rather than being hidden inside a conversion step.

## 10. Inspect the Affected `TotalCharges` Records

Before deciding how to handle the missing values, inspect their context.

In [9]:
totalcharges_missing = df.loc[df["TotalCharges"].isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]  # Select the affected records and relevant context.
display(totalcharges_missing)  # Display every affected record for review.

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,NaN,Two year,No
753,3115-CZMZD,0,20.25,NaN,Two year,No
936,5709-LVOEQ,0,80.85,NaN,Two year,No
1082,4367-NUYAO,0,25.75,NaN,Two year,No
1340,1371-DWPAZ,0,56.05,NaN,Two year,No
3331,7644-OMVMY,0,19.85,NaN,Two year,No
3826,3213-VVOLG,0,25.35,NaN,Two year,No
4380,2520-SGTTA,0,20.00,NaN,Two year,No
5218,2923-ARZLG,0,19.70,NaN,One year,No
6670,4075-WKNIU,0,73.35,NaN,Two year,No


### What was done and why

The missing records are reviewed alongside `tenure`, `MonthlyCharges`, `Contract` and `Churn`.

The upstream assessment found that all 11 affected records have `tenure = 0` and `Churn = No`.

This pattern supports treating the missing cumulative charge as a special early-tenure condition, but the preparation team must still make the final treatment explicit and validate the effect.

## 11. Define the `TotalCharges` Treatment

For this classroom dataset, the preparation decision is to represent the tenure-zero missing cumulative charge as zero, after documenting the rationale.

In [10]:
zero_charge_mask = df["TotalCharges"].isna() & df["tenure"].eq(0)  # Identify missing cumulative charges occurring at zero tenure.
print("Missing TotalCharges with zero tenure:", int(zero_charge_mask.sum()))  # Confirm the expected affected population.
df.loc[zero_charge_mask, "TotalCharges"] = 0.0  # Represent the initial zero-tenure cumulative charge as zero.

Missing TotalCharges with zero tenure: 11


### What was done and why

The decision is based on the observed raw-data pattern: the 11 missing `TotalCharges` records all have zero tenure.

The important point is not simply the replacement itself; it is that the replacement is **explicit, conditional and documented**.

We are not saying that every missing `TotalCharges` value should always be zero. We are saying that, for this supplied dataset, the identified missing values correspond to zero-tenure records and are therefore represented as zero for preparation.

Any future dataset with a different pattern must be reassessed rather than blindly reusing this assumption.

## 12. Validate `TotalCharges` After Treatment

Confirm that the intended missing values have been resolved and no unexpected missing values remain.

In [11]:
print("Remaining TotalCharges missing:", int(df["TotalCharges"].isna().sum()))  # Count any remaining missing cumulative charges.
print("TotalCharges dtype:", df["TotalCharges"].dtype)  # Confirm the field remains numeric.
display(df["TotalCharges"].describe().to_frame("TotalCharges summary"))  # Inspect the prepared numerical distribution.

Remaining TotalCharges missing: 0
TotalCharges dtype: float64


,TotalCharges summary
count,7043.000000
mean,2279.734304
std,2266.794470
min,0.000000
25%,398.550000
50%,1394.550000
75%,3786.600000
max,8684.800000


### What was done and why

This is a validation checkpoint.

We expect:

- zero remaining missing `TotalCharges` values;
- a numeric datatype;
- sensible numerical summary statistics.

The check ensures the treatment was actually applied and did not introduce another issue.

## 13. Validate Numerical Fields

The main numerical fields are checked for invalid missing values and unexpected datatypes.

In [12]:
numeric_expected = ["tenure", "MonthlyCharges", "TotalCharges"]  # Define the fields that should be numerical for modelling.
for column in numeric_expected:  # Inspect each expected numerical field.
    print(f"{column}: dtype={df[column].dtype}, missing={df[column].isna().sum()}")  # Report datatype and missing count.

tenure: dtype=int64, missing=0
MonthlyCharges: dtype=float64, missing=0
TotalCharges: dtype=float64, missing=0


### What was done and why

The preparation stage establishes an explicit numerical feature list.

`tenure`, `MonthlyCharges` and `TotalCharges` should all be numeric before they enter the numerical preprocessing pipeline.

The loop makes it easy to catch a future dataset-version mismatch.

## 14. Validate Categorical Values

The raw category vocabulary is inspected before encoding.

In [13]:
categorical_expected = [  # Define the expected categorical feature fields.
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService",  # Include profile and core-service fields.
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",  # Include ancillary-service fields.
    "Contract", "PaperlessBilling", "PaymentMethod"  # Include account and billing fields.
]  # Finish the categorical feature list.
for column in categorical_expected:  # Inspect each categorical feature.
    print(f"\n--- {column} ---")  # Print a readable field heading.
    print(df[column].unique())  # Display the raw category values.


--- gender ---
['Female' 'Male']

--- Partner ---
['Yes' 'No']

--- Dependents ---
['No' 'Yes']

--- PhoneService ---
['No' 'Yes']

--- MultipleLines ---
['No phone service' 'No' 'Yes']

--- InternetService ---
['DSL' 'Fiber optic' 'No']

--- OnlineSecurity ---
['No' 'Yes' 'No internet service']

--- OnlineBackup ---
['Yes' 'No' 'No internet service']

--- DeviceProtection ---
['No' 'Yes' 'No internet service']

--- TechSupport ---
['No' 'Yes' 'No internet service']

--- StreamingTV ---
['No' 'Yes' 'No internet service']

--- StreamingMovies ---
['No' 'Yes' 'No internet service']

--- Contract ---
['Month-to-month' 'One year' 'Two year']

--- PaperlessBilling ---
['Yes' 'No']

--- PaymentMethod ---
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


### What was done and why

Categorical fields must be understood before encoding.

The preparation pipeline should preserve meaningful structural categories such as:

- `No phone service`
- `No internet service`

These categories are not automatically interchangeable with ordinary `No`.

The raw vocabulary is therefore inspected before One-Hot Encoding is applied.

## 15. Separate Target From Features

The target is removed from the feature matrix and encoded separately.

In [14]:
X = df.drop(columns=["Churn"])  # Create the feature table without the target column.
y = df["Churn"].map({"No": 0, "Yes": 1})  # Convert the binary target into numerical labels for model training.
print("Feature shape:", X.shape)  # Display the feature-table dimensions.
print("Target shape:", y.shape)  # Display the target dimensions.
display(y.value_counts().sort_index().to_frame("Target count"))  # Confirm the target encoding.

Feature shape: (7043, 20)
Target shape: (7043,)


,Target count
Churn,
0,5174
1,1869


### What was done and why

This is a fundamental supervised-learning separation:

- `X` contains information the model may use.
- `y` contains the answer the model must learn.

The mapping is:

- `No → 0`
- `Yes → 1`

The target is encoded separately so it is never accidentally transformed as if it were an ordinary customer feature.

## 16. Remove the Identifier From Predictive Features

`customerID` is preserved outside the model feature matrix for traceability.

In [15]:
customer_ids = X["customerID"].copy()  # Preserve customer IDs separately for traceability and later application output.
X = X.drop(columns=["customerID"])  # Remove the identifier from the predictive feature matrix.
print("customerID retained separately:", len(customer_ids) == len(X))  # Confirm traceability records still align with observations.
print("Feature columns after identifier removal:", X.columns.tolist())  # Display the actual predictive fields.

customerID retained separately: True
Feature columns after identifier removal: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


### What was done and why

`customerID` identifies a customer but does not describe customer behaviour.

Including it would allow the model to learn arbitrary identifier patterns that have no defensible business meaning.

The identifier is therefore:

- retained separately for traceability;
- excluded from `X`;
- not passed through encoding or scaling.

This keeps the model focused on customer attributes rather than arbitrary IDs.

## 17. Define Numerical and Categorical Feature Groups

The preprocessing pipeline needs explicit feature groups.

In [16]:
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]  # Define the numerical features requiring numerical preprocessing.
categorical_features = [column for column in X.columns if column not in numerical_features]  # Treat all remaining predictive fields as categorical features.
print("Numerical features:", numerical_features)  # Display the numerical feature group.
print("Categorical features:", categorical_features)  # Display the categorical feature group.

Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### What was done and why

Explicit feature grouping is important because different data types require different transformations.

### Numerical features

- `tenure`
- `MonthlyCharges`
- `TotalCharges`

### Categorical features

All remaining predictive fields are categorical.

This separation allows the pipeline to apply imputation/scaling to numerical variables and imputation/encoding to categorical variables.

## 18. Create the Numerical Preprocessing Pipeline

Numerical preprocessing is defined as a reusable pipeline.

In [17]:
numeric_pipeline = Pipeline(steps=[  # Create a reproducible sequence for numerical features.
    ("imputer", SimpleImputer(strategy="median")),  # Replace any future numerical missing values with the training-set median.
    ("scaler", StandardScaler())  # Standardise numerical features using statistics learned from training data.
])  # Finish the numerical pipeline.

### What was done and why

The numerical pipeline contains two stages.

**1. Median imputation**

This is a defensive step. The current prepared data has no missing numerical values, but the pipeline remains robust if a future training split contains missing values.

**2. StandardScaler**

Scaling places numerical features on a comparable scale.

Most importantly, the scaler is fitted only when the training data is passed through the pipeline. This prevents test-set information from influencing training transformations.

## 19. Create the Categorical Preprocessing Pipeline

Categorical preparation is also made reproducible.

In [18]:
categorical_pipeline = Pipeline(steps=[  # Create a reproducible sequence for categorical features.
    ("imputer", SimpleImputer(strategy="most_frequent")),  # Fill unexpected categorical missing values using the training-set most frequent category.
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))  # Convert categories to binary features while safely handling unseen future categories.
])  # Finish the categorical pipeline.

### What was done and why

Categorical fields cannot be passed directly to most scikit-learn classifiers.

The pipeline therefore:

1. provides defensive missing-value handling;
2. one-hot encodes the categories.

`handle_unknown="ignore"` is important for the future prototype. A new customer may have a category that was not observed during model training. Instead of crashing, the encoder can handle the unseen category safely.

The raw categories remain conceptually meaningful even though the model receives numerical indicator columns.

## 20. Combine the Numerical and Categorical Pipelines

`ColumnTransformer` applies the correct pipeline to each feature group.

In [19]:
preprocessor = ColumnTransformer(  # Create one preprocessing object for the complete feature table.
    transformers=[  # Define the transformations and the columns they apply to.
        ("numerical", numeric_pipeline, numerical_features),  # Apply the numerical pipeline to numerical fields.
        ("categorical", categorical_pipeline, categorical_features)  # Apply the categorical pipeline to categorical fields.
    ],
    remainder="drop"  # Drop any field not explicitly included in the approved feature groups.
)  # Finish the combined preprocessing object.

### What was done and why

This is the central preparation object.

`ColumnTransformer` ensures that:

- numerical fields receive numerical treatment;
- categorical fields receive categorical treatment;
- unexpected fields are not silently passed into the model.

The result is one reproducible preprocessing object that downstream model development can place inside a model pipeline.

## 21. Create the Train/Test Split Before Fitting Preprocessing

The test set must remain independent so that it does not influence fitted preprocessing statistics.

In [20]:
X_train, X_test, y_train, y_test = train_test_split(  # Split features and target into independent training and test sets.
    X,  # Supply the feature matrix.
    y,  # Supply the encoded target.
    test_size=0.20,  # Reserve 20 percent of the data for final holdout testing.
    random_state=42,  # Fix the random seed so the split is reproducible.
    stratify=y  # Preserve approximately the same churn class proportions in train and test sets.
)  # Finish the train/test split.
print("Training features:", X_train.shape)  # Display training feature dimensions.
print("Test features:", X_test.shape)  # Display test feature dimensions.
print("Training target distribution:")  # Label the training target output.
print(y_train.value_counts(normalize=True).sort_index())  # Display training class proportions.
print("Test target distribution:")  # Label the test target output.
print(y_test.value_counts(normalize=True).sort_index())  # Display test class proportions.

Training features: (5634, 19)
Test features: (1409, 19)
Training target distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Test target distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


### What was done and why

The split occurs **before fitting the preprocessing object**.

This is a crucial leakage-prevention step.

If scaling or imputation statistics were calculated from the entire dataset before splitting, information from the test set could influence the training process.

`stratify=y` is used because churn classes are not evenly distributed. It helps preserve comparable class proportions in both partitions.

The test set should remain untouched until model evaluation.

## 22. Fit the Preprocessor on Training Data Only

The preprocessing object learns its parameters exclusively from the training set.

In [21]:
X_train_prepared = preprocessor.fit_transform(X_train)  # Fit all preprocessing rules using training data and transform the training features.
X_test_prepared = preprocessor.transform(X_test)  # Apply the already-fitted rules to the test features without refitting.
print("Prepared training shape:", X_train_prepared.shape)  # Report the transformed training dimensions.
print("Prepared test shape:", X_test_prepared.shape)  # Report the transformed test dimensions.

Prepared training shape: (5634, 46)
Prepared test shape: (1409, 46)


### What was done and why

This is the core leakage-control point.

`fit_transform(X_train)`:

- learns medians for numerical imputation;
- learns scaling statistics;
- learns categorical vocabulary;
- transforms training data.

`transform(X_test)`:

- uses those already-learned rules;
- does not learn anything from the test set.

This ensures that the test set remains an independent estimate of later model performance.

## 23. Inspect the Generated Feature Names

One-hot encoding expands categorical variables into numerical columns. We inspect the resulting feature names.

In [22]:
feature_names = preprocessor.get_feature_names_out()  # Retrieve the names generated by the complete preprocessing pipeline.
print("Number of prepared features:", len(feature_names))  # Report the total transformed feature count.
display(pd.Series(feature_names, name="Prepared feature"))  # Display the transformed feature names.

Number of prepared features: 46


,Prepared feature
0,numerical__tenure
1,numerical__MonthlyCharges
2,numerical__TotalCharges
3,categorical__gender_Female
4,categorical__gender_Male
5,categorical__SeniorCitizen_0
6,categorical__SeniorCitizen_1
7,categorical__Partner_No
8,categorical__Partner_Yes
9,categorical__Dependents_No


### What was done and why

One-hot encoding changes the dimensionality of the dataset because each categorical level becomes a machine-readable feature.

Inspecting the feature names is important for:

- debugging;
- interpreting model coefficients or feature importance later;
- verifying that expected categories were encoded;
- detecting accidental leakage or unexpected fields.

This gives the Model Development Lead a clear understanding of what the model will actually receive.

## 24. Convert Prepared Arrays to DataFrames for Inspection

The transformed arrays are temporarily represented as DataFrames so their structure can be inspected easily.

In [23]:
X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names, index=X_train.index)  # Build a readable training DataFrame from the transformed array.
X_test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names, index=X_test.index)  # Build a readable test DataFrame from the transformed array.
display(X_train_prepared_df.head())  # Inspect several prepared training records.
display(X_test_prepared_df.head())  # Inspect several prepared test records.

,numerical__tenure,numerical__MonthlyCharges,numerical__TotalCharges,categorical__gender_Female,categorical__gender_Male,categorical__SeniorCitizen_0,categorical__SeniorCitizen_1,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,...,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check
3738,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3151,-0.711743,0.337478,-0.503635,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4860,-0.793155,-0.809013,-0.749883,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3867,-0.263980,0.284384,-0.172722,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3810,-1.281624,-0.676279,-0.989374,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


,numerical__tenure,numerical__MonthlyCharges,numerical__TotalCharges,categorical__gender_Female,categorical__gender_Male,categorical__SeniorCitizen_0,categorical__SeniorCitizen_1,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,...,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check
437,1.608483,1.629976,2.706828,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2280,-0.996684,1.168725,-0.610260,1.0,0.0,0.0,1.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2235,0.346606,0.445324,0.400116,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4460,-0.589626,0.440347,-0.364451,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3761,1.608483,0.588013,1.588421,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


### What was done and why

The model can work with arrays, but DataFrames make the transformed structure easier for humans to audit.

The original row indexes are retained so the transformed records can be traced back during debugging without using `customerID` as a predictive feature.

## 25. Validate Prepared Data for Missing Values

The final feature matrix should not contain missing numerical values.

In [24]:
train_missing = int(np.isnan(X_train_prepared).sum())  # Count NaN values in the prepared training matrix.
test_missing = int(np.isnan(X_test_prepared).sum())  # Count NaN values in the prepared test matrix.
print("Prepared training NaN values:", train_missing)  # Report training missing values.
print("Prepared test NaN values:", test_missing)  # Report test missing values.
assert train_missing == 0, "Prepared training data contains NaN values."  # Fail if training data is still incomplete.
assert test_missing == 0, "Prepared test data contains NaN values."  # Fail if test data is still incomplete.

Prepared training NaN values: 0
Prepared test NaN values: 0


### What was done and why

The preparation stage should produce a model-ready numerical matrix without unresolved missing values.

The assertions turn the expectation into an automated quality gate.

If either assertion fails, the team should stop and investigate instead of passing defective data into model development.

## 26. Validate Prepared Data Contains Only Numeric Values

The complete transformed matrix must be numerical.

In [25]:
print("Training matrix dtype:", X_train_prepared.dtype)  # Report the underlying training matrix datatype.
print("Test matrix dtype:", X_test_prepared.dtype)  # Report the underlying test matrix datatype.
assert np.issubdtype(X_train_prepared.dtype, np.number), "Training matrix is not numeric."  # Confirm training data is numerical.
assert np.issubdtype(X_test_prepared.dtype, np.number), "Test matrix is not numeric."  # Confirm test data is numerical.

Training matrix dtype: float64
Test matrix dtype: float64


### What was done and why

Machine-learning estimators generally require numerical inputs.

At the end of preparation, the categorical variables have been encoded and the numerical variables have been processed, producing a numerical feature matrix suitable for model development.

## 27. Validate the Target Encoding

The target should contain only the expected binary values.

In [26]:
print("Unique encoded target values:", sorted(y.unique()))  # Display all unique target labels after encoding.
assert set(y.unique()).issubset({0, 1}), "Unexpected target values detected."  # Stop if the target contains anything other than 0 or 1.

Unique encoded target values: [np.int64(0), np.int64(1)]


### What was done and why

The target is a binary classification problem.

The assertion protects against unexpected labels such as spelling variations, blanks or additional categories appearing in a future dataset version.

## 28. Compare Train/Test Row Counts

We confirm that the split accounts for every original prepared observation.

In [27]:
total_split_rows = len(X_train) + len(X_test)  # Add training and test observations.
print("Original feature rows:", len(X))  # Display the number of rows before splitting.
print("Rows after train/test split:", total_split_rows)  # Display the combined split size.
assert total_split_rows == len(X), "Train/test split does not account for all rows."  # Confirm no observations were lost.

Original feature rows: 7043
Rows after train/test split: 7043


### What was done and why

The train/test split should partition the dataset, not accidentally delete observations.

This check confirms that every prepared observation belongs to exactly one of the two partitions.

## 29. Validate Target Alignment

Feature rows and target rows must remain aligned after splitting.

In [28]:
assert len(X_train) == len(y_train), "Training features and target are misaligned."  # Confirm training row counts match.
assert len(X_test) == len(y_test), "Test features and target are misaligned."  # Confirm test row counts match.
print("Training alignment: PASS")  # Report successful training alignment.
print("Test alignment: PASS")  # Report successful test alignment.

Training alignment: PASS
Test alignment: PASS


### What was done and why

A model requires every feature row to correspond to the correct target label.

These assertions provide a simple but important safeguard against accidental indexing or filtering errors.

## 30. Save the Prepared Data for Downstream Development

Prepared arrays and targets are saved as derived artefacts rather than replacing the raw dataset.

In [29]:
prepared_train = pd.DataFrame(X_train_prepared, columns=feature_names)  # Store prepared training features in a DataFrame.
prepared_test = pd.DataFrame(X_test_prepared, columns=feature_names)  # Store prepared test features in a DataFrame.
prepared_train["Churn"] = y_train.to_numpy()  # Add the encoded training target as a separate output field.
prepared_test["Churn"] = y_test.to_numpy()  # Add the encoded test target as a separate output field.
prepared_train.to_csv("prepared_train.csv", index=False)  # Save the prepared training dataset as a derived artefact.
prepared_test.to_csv("prepared_test.csv", index=False)  # Save the prepared test dataset as a derived artefact.
print("Prepared training file saved.")  # Confirm the training export.
print("Prepared test file saved.")  # Confirm the test export.

Prepared training file saved.
Prepared test file saved.


### What was done and why

The prepared datasets are saved as **derived artefacts**.

The raw `Customer_churn.csv` remains untouched.

These CSVs are useful for inspection and downstream work, but the more important artefact is the fitted preprocessing object because it defines the transformation logic.

For the final project repository, these files should be versioned according to the team's agreed data-management policy. If the repository should not contain generated datasets, document their canonical storage location instead.

## 31. Save the Preprocessing Pipeline

The fitted preprocessing object is saved so that the exact training transformation can be reused.

In [30]:
import joblib  # Import joblib for serialising the fitted preprocessing object.
joblib.dump(preprocessor, "preprocessor.joblib")  # Save the fitted preprocessing pipeline to disk.
print("Preprocessing pipeline saved as preprocessor.joblib")  # Confirm the saved artefact.

Preprocessing pipeline saved as preprocessor.joblib


### What was done and why

Saving the fitted preprocessing object is critical for reproducibility.

The prototype must process new customer information using the **same transformations used during training**.

Without the saved pipeline, a future developer might accidentally:

- calculate new scaling statistics;
- use different category mappings;
- handle missing values differently;
- or transform fields in a different order.

That would make the deployed system inconsistent with the trained model.

## 32. Reload the Pipeline as a Reproducibility Test

The saved pipeline is reloaded to confirm that the artefact can be reused.

In [31]:
loaded_preprocessor = joblib.load("preprocessor.joblib")  # Reload the saved preprocessing pipeline from disk.
reloaded_test = loaded_preprocessor.transform(X_test)  # Transform the test data using the reloaded pipeline.
print("Reloaded transformed test shape:", reloaded_test.shape)  # Confirm the reloaded pipeline produces the expected shape.
assert reloaded_test.shape == X_test_prepared.shape, "Reloaded pipeline produced an unexpected shape."  # Validate consistency with the original transformation.

Reloaded transformed test shape: (1409, 46)


### What was done and why

This is a basic reproducibility test.

A pipeline that works only while the current Python session is alive is not enough for a real project.

Reloading the saved object verifies that the preprocessing artefact can be persisted and used again.

The same concept will later be used by the Application/Integration Lead when the final model is integrated into the prototype.

## 33. Produce a Preparation Summary

The final preparation state is summarised for the handoff.

In [40]:
preparation_summary = pd.DataFrame({  # Create a concise technical handoff table.
    "Item": [  # Define the preparation checkpoints.
        "Raw rows", "Raw columns", "Exact duplicate rows", "Duplicate customer IDs", "Whitespace TotalCharges handled",
        "TotalCharges numeric", "Identifier removed from features", "Train/test split", "Prepared training rows",
        "Prepared test rows", "Prepared feature count", "Prepared NaN values"
    ],  # Finish the checkpoint labels.
    "Result": [  # Record the observed outcomes.
        len(raw_df), len(raw_df.columns), duplicate_rows, duplicate_ids, blanks_tc,
        str(df["TotalCharges"].dtype), "Yes", "80/20 stratified", len(X_train),
        len(X_test), len(feature_names), train_missing + test_missing
    ]  # Finish the results.
})  # Build the summary DataFrame.
display(preparation_summary)  # Display the final preparation summary.

NameError: name 'blanks_tc' is not defined

### What was done and why

This summary provides the Model Development Lead with a compact view of the preparation state.

The model-development handoff should include:

- the prepared training and test data;
- the target definition;
- the feature list;
- the fitted preprocessing pipeline;
- the preparation decisions;
- and the validation results.

The Model Development Lead should not have to reverse-engineer how the data was prepared.

# 34. Data Preparation Decision Log

## Decision 1 — Preserve the raw dataset

**Decision:** Do not overwrite `Customer_churn.csv`.

**Reason:** The raw source is required for reproducibility and auditability.

## Decision 2 — Handle whitespace-only `TotalCharges`

**Decision:** Strip whitespace, convert empty strings to missing, convert the field to numeric, then represent the identified zero-tenure missing values as `0.0`.

**Reason:** All 11 affected records have `tenure = 0`, and the cumulative-charge field therefore needs an explicit treatment before numerical modelling.

**Caveat:** This assumption is specific to the observed dataset pattern and should not be blindly applied to a different dataset.

## Decision 3 — Exclude `customerID` from model features

**Decision:** Preserve it separately for traceability but remove it from `X`.

**Reason:** It is an identifier, not a meaningful behavioural predictor.

## Decision 4 — One-hot encode categorical variables

**Decision:** Use `OneHotEncoder(handle_unknown="ignore")`.

**Reason:** The classification models require numerical inputs, while the categorical semantics should be retained without imposing artificial numeric ordering.

## Decision 5 — Scale numerical variables

**Decision:** Use `StandardScaler` inside the numerical preprocessing pipeline.

**Reason:** This provides standardised numerical inputs and ensures scaling statistics are learned only from training data.

## Decision 6 — Split before fitting preprocessing

**Decision:** Create a stratified 80/20 train/test split and fit preprocessing only on the training partition.

**Reason:** This prevents information leakage from the test set into training transformations.

## Decision 7 — Use a reproducible pipeline

**Decision:** Combine transformations with `ColumnTransformer` and `Pipeline`, then save the fitted preprocessing object.

**Reason:** Training and future inference must use identical preprocessing rules.

# 35. Handoff to Model Development

### Prepared-data handoff

The Data Preparation stage hands the following to the Model Development Lead:

1. `X_train` / prepared training features.
2. `X_test` / prepared test features.
3. `y_train`.
4. `y_test`.
5. `feature_names`.
6. `preprocessor.joblib`.
7. `prepared_train.csv`.
8. `prepared_test.csv`.
9. This notebook documenting every preparation decision.

### Model Development must NOT assume

- that `customerID` is a predictive feature;
- that the raw `TotalCharges` column is ready for modelling;
- that preprocessing can be manually recreated;
- that the test set may be used to tune preprocessing;
- or that the preparation decisions are undocumented.

### Required downstream practice

The Model Development Lead should place the **preprocessor and model inside one final modelling pipeline** where practical, so that a raw customer input can follow:

**raw customer fields → preprocessing → trained model → churn prediction**

This keeps the prototype consistent with the training process.

# 36. Final Data Preparation Stage Gate

Before declaring this stage complete, the team should verify:

- [ ] Raw source remains unchanged.
- [ ] Dataset dimensions have been validated.
- [ ] Duplicate rows have been checked.
- [ ] Duplicate IDs have been checked.
- [ ] `TotalCharges` whitespace has been handled.
- [ ] `TotalCharges` has been converted to numeric.
- [ ] The 11 zero-tenure records have an explicit treatment.
- [ ] `customerID` is excluded from predictive features.
- [ ] Numerical and categorical feature groups are explicitly defined.
- [ ] Categorical features are encoded reproducibly.
- [ ] Numerical features are processed reproducibly.
- [ ] Train/test splitting is stratified.
- [ ] Preprocessing is fitted only on training data.
- [ ] Test data is transformed without refitting.
- [ ] Prepared data contains no unresolved NaNs.
- [ ] Prepared features are numerical.
- [ ] Target values are valid binary labels.
- [ ] Prepared datasets are aligned with their targets.
- [ ] The preprocessing pipeline can be saved and reloaded.
- [ ] The team has reviewed and understood the preparation decisions.
- [ ] Model Development Lead has accepted the handoff.

### Completion rule

**Data Preparation is complete when the team can explain not only what the final prepared data looks like, but why every transformation was performed and how the exact same transformation will be reproduced later.**